## Extracting Data From Github

<a href="https://colab.research.google.com/github/anaclaraaraujo/github_reaction_analysis/blob/main/extracting_data_from_github.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Descrição do Código  
Este script realiza a **coleta e análise de issues e pull requests** de repositórios pertencentes a uma organização no **GitHub**. Ele filtra os itens que possuem as **labels "bug" ou "enhancement"** e que receberam **reações** dos usuários (como 👍, 🎉, ❤️, etc.).  

Os dados extraídos são salvos em um **arquivo CSV no Google Drive**, permitindo análises futuras sobre a interação da comunidade com esses repositórios.  


### Funcionalidades  

#### 1️⃣ **Autenticação e Configuração**  
- Utiliza um **GitHub Token** para acessar a API do GitHub.  
- Define a organização-alvo (**"elastic"** no exemplo).  
- Especifica as **labels** desejadas ("bug" e "enhancement").  

#### 2️⃣ **Coleta de Dados**  
- Busca **todos os repositórios** da organização especificada.  
- Para cada repositório, obtém **issues e pull requests**.  
- Filtra apenas aqueles que possuem as labels especificadas **e** reações dos usuários.  

#### 3️⃣ **Controle de Limite de Requisições**  
- A API do GitHub tem um **limite de requisições** por hora.  
- A função `check_rate_limit()` monitora esse limite e aguarda automaticamente caso seja atingido.  

#### 4️⃣ **Armazenamento dos Dados**  
- Os dados são organizados em um **DataFrame do Pandas**.  
- O arquivo CSV gerado é salvo no **Google Drive** no caminho especificado.  


### 📊 **Estrutura dos Dados Salvos**  
O arquivo CSV contém as seguintes colunas:  

| Repositório | URL | Linguagem | Estrelas | ID | Número | Título | Estado | Comentários | Criado em | Atualizado em | Labels | Reações (+1, 🎉, ❤️, etc.) |
|------------|----|-----------|----------|----|-------|--------|-------|-------------|------------|--------------|--------|-------------------|
| repo-name  | github.com/repo | Python | 1200 | 123456 | 42 | Issue title | open | 5 | 2023-01-01 | 2023-01-02 | ['bug'] | {5, 2, 1, ...} |

Cada linha representa uma **issue ou pull request** que atendeu aos critérios.  

# 1️⃣ Montar o Google Drive

In [ ]:
print("📂 Montando Google Drive...")
from google.colab import drive
drive.mount('/content/drive')

# 2️⃣ Configurar Token e Variáveis

In [ ]:
import requests
import pandas as pd
import time
import os
from datetime import datetime
import json
from concurrent.futures import ThreadPoolExecutor

# 🔹 Configurar o Token do GitHub
GITHUB_TOKEN = "GITHUB_API_KEY"
HEADERS = {"Authorization": f"token {GITHUB_TOKEN}"}
SAVE_INTERVAL = 500

# 🔹 Configurações gerais
ORG_NAME = "elastic"
GITHUB_API_URL = f"https://api.github.com/orgs/{ORG_NAME}/repos"
labels_filter = ['bug', 'enhancement']

# 3️⃣ Definir Funções Auxiliares

In [ ]:
def check_rate_limit():
    response = requests.get("https://api.github.com/rate_limit", headers=HEADERS)
    if response.status_code == 200:
        rate_limit_data = response.json()
        remaining = rate_limit_data['resources']['core']['remaining']
        reset_time = rate_limit_data['resources']['core']['reset']
        print(f"🔄 Limite de requisições restantes: {remaining}")
        if remaining == 0:
            reset_timestamp = reset_time - time.time()
            print(f"⚠️ Limite de requisições atingido. Aguardando {reset_timestamp:.0f} segundos.")
            time.sleep(reset_timestamp + 10)
        return remaining
    return 0

def has_bug_or_enhancement(labels):
    return any(label['name'].lower() in labels_filter for label in labels)

def get_filtered_items(repo_owner, repo_name, item_type):
    url = f"https://api.github.com/repos/{repo_owner}/{repo_name}/{item_type}?state=all"
    print(f"🔍 Buscando {item_type} em {repo_owner}/{repo_name}...")
    response = requests.get(url, headers=HEADERS)
    if response.status_code == 200:
        try:
            data = response.json()
            return [item for item in data if has_bug_or_enhancement(item.get('labels', []))]
        except ValueError:
            print(f"❌ Erro ao processar JSON da URL: {url}")
    return []
from datetime import datetime

# 4️⃣ Coletar e Processar Repositórios

In [ ]:
repos_data = []
page = 1

while True:
    check_rate_limit()
    print(f"📡 Buscando repositórios - Página {page}...")
    response = requests.get(f"{GITHUB_API_URL}?per_page=100&page={page}", headers=HEADERS)

    if response.status_code != 200 or not response.json():
        print("🚫 Nenhum repositório encontrado ou erro na requisição.")
        break

    for repo in response.json():
        repo_lang = repo["language"].lower() if repo["language"] else ""
        repo_owner, repo_name = ORG_NAME, repo["name"]
        print(f"📌 Processando repositório: {repo_name} ({repo_lang}) - {repo['html_url']}")

        check_rate_limit()
        issues = get_filtered_items(repo_owner, repo_name, "issues")
        pulls = get_filtered_items(repo_owner, repo_name, "pulls")

        for item in issues + pulls:
            reactions = item.get("reactions", {})

            if any(reactions.get(r, 0) > 0 for r in ["+1", "-1", "laugh", "hooray", "confused", "heart", "rocket", "eyes"]):
                repos_data.append({
                    "repository": repo["name"],
                    "url_repo": repo["html_url"],
                    "url": item["html_url"],
                    "language": repo_lang,
                    "stars": repo["stargazers_count"],
                    "id": item["id"],
                    "number": item["number"],
                    "title": item["title"],
                    "body": item["body"],
                    "state": item["state"],
                    "comments": item.get("comments", 0),
                    "created_at": item["created_at"],
                    "updated_at": item["updated_at"],
                    "closed_at": item.get("closed_at"),
                    "labels": [label["name"] for label in item.get("labels", [])],
                    "plus_one": reactions.get("+1", 0),
                    "plus_minus": reactions.get("-1", 0),
                    "laugh": reactions.get("laugh", 0),
                    "hooray": reactions.get("hooray", 0),
                    "confused": reactions.get("confused", 0),
                    "heart": reactions.get("heart", 0),
                    "rocket": reactions.get("rocket", 0),
                    "eyes": reactions.get("eyes", 0)
                })
                print(f"✅ Adicionado: {item['title']} ({'PR' if 'pull_request' in item else 'Issue'})")
            else:
                print(f"❌ Ignorado (sem reações): {item['title']}")

    page += 1
    time.sleep(5)

# 5️⃣ Criar DataFrame e Salvar no Google Drive

In [ ]:
if repos_data:
    df = pd.DataFrame(repos_data)
    caminho_arquivo = f"/content/drive/My Drive/database/{ORG_NAME}.csv"
    df.to_csv(caminho_arquivo, index=False, encoding="utf-8")
    print(f"✅ Arquivo salvo com sucesso em: {caminho_arquivo}")
else:
    print("⚠ Nenhuma issue ou pull request com reações encontrada.")